In [ ]:
import math
from typing import TypedDict

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from pingouin import mwu
from statsmodels.stats.multitest import multipletests

In [ ]:
mpl.rcParams["font.family"] = "DejaVu Sans"

In [ ]:
class PlotConfig(TypedDict):
    title: str
    xlabel: str


def format_pval(p: float) -> str:
    """Format a p-value the way the manuscript prints it."""
    if p < 0.001:
        return "$p$ < 0.001"
    elif p < 0.05:
        return f"$p$ = {p:.3f}"
    else:
        return "ns"


def generate_violin_panels(
    results_df: pd.DataFrame,
    statistics_df: pd.DataFrame,
    plot_config: dict[str, PlotConfig],
    save_path: str,
    cut: float | list[float] = 0,
    panel_start: int = 0,
):
    """
    Generate multi-panel violin plots with Mann-Whitney U tests
    in Nature journal style.

    Adds a single shared legend at the bottom with the group colors.
    """
    group_column = "Interviewer"

    sns.set_theme(style="whitegrid")
    sns.set_context("paper", font_scale=1.2)

    # --- Consistent group order & palette across all panels ---
    # Use the categorical order if present; otherwise, use sorted unique values.
    groups = list(reversed(sorted(results_df[group_column].dropna().unique().tolist())))

    palette_list = sns.color_palette("deep", n_colors=len(groups))
    palette = dict(zip(groups, palette_list))

    variables = list(plot_config.keys())
    n_panels = len(variables)

    if n_panels <= 2:
        n_rows, n_cols = 1, n_panels
    else:
        n_rows, n_cols = math.ceil(n_panels / 2), 2
    # Keep each panel the same height whatever the panel count, so the figure
    # scales to one column width without the rows squashing.
    figsize = (10, 4 if n_rows == 1 else 1.1 + 2.9 * n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, sharey=False, squeeze=False)
    axes = axes.flatten()
    n_used = min(n_panels, len(axes))
    plot_axes = axes[:n_used]

    # Hide unused axes
    for ax in axes[n_used:]:
        ax.set_visible(False)

    if not isinstance(cut, list):
        cut = [cut] * n_used

    for i, (ax, var, c) in enumerate(zip(plot_axes, variables, cut)):
        cfg = plot_config[var]

        if len(groups) >= 2:
            p = statistics_df.loc[cfg["title"]]["p-val"]
            p_text = format_pval(p)
        else:
            p_text = "n/a"

        if var in {"avg_response_duration_by_interviewer(sec)", "specificity", "relevance"}:
            # Hist panels
            # Combine data range across all groups
            all_values = results_df[var].dropna()
            min_val, max_val = all_values.min(), all_values.max()

            # Define shared bins based on global range
            if var in {"specificity", "relevance"}:
                bins = np.linspace(1, 5, 30)
            else:
                bins = np.linspace(min_val, max_val, 30)

            # Plot each group using the same bins and consistent colors
            for grp in groups:
                subset = results_df[results_df[group_column] == grp]
                ax.hist(
                    subset[var],
                    bins=bins,
                    alpha=0.6,
                    label=grp,
                    color=palette[grp],
                    edgecolor="black",
                    linewidth=0.8,
                )
                ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

            # Overlay median lines
            for grp in groups:
                med = results_df[results_df[group_column] == grp][var].median()
                ax.axvline(
                    med,
                    color=palette[grp],
                    linestyle="--",
                    linewidth=1.2,
                    alpha=0.9,
                    zorder=3,
                )

            ax.set_ylabel("Count")

        else:
            # Main violin + box + swarm with consistent palette
            sns.violinplot(
                data=results_df,
                x=var,
                y=group_column,
                ax=ax,
                hue=group_column,
                palette=palette,
                inner="box",
                inner_kws=dict(box_width=10, whis_width=2),
                linewidth=1.2,
                alpha=0.9,
                cut=c,
            )
            sns.swarmplot(
                data=results_df,
                x=var,
                y=group_column,
                ax=ax,
                color="black",
                alpha=0.6,
            )
            ax.set_ylabel("")

        # Panel label (A, B, C, D, E...)
        ax.text(
            -0.15,
            1.05,
            chr(65 + panel_start + i),
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            va="center",
            ha="center",
        )

        # Titles and labels
        ax.set_title(cfg["title"], fontsize=12, fontweight="bold", pad=8)
        ax.set_xlabel(cfg["xlabel"], fontsize=10)

        # Remove any per-axes legends; we'll add a shared one at the bottom
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # p-value annotation
        ax.text(
            0.98,
            0.5,
            p_text,
            transform=ax.transAxes,
            fontsize=10,
            ha="right",
            va="center",
            color="black",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2),
        )

        sns.despine(ax=ax, top=True, right=True)
        ax.set_facecolor("white")

    # --- Shared legend at the bottom ---
    handles = [
        mpatches.Patch(facecolor=palette[g], edgecolor="black", label=g)
        for g in groups
    ]
    # Reserve enough figure-level space for the shared legend.
    fig.tight_layout(h_pad=1.5, w_pad=1.5)
    # Reserve a strip tall enough for the bottom row's x-label *and* the
    # shared legend beneath it, or the two overlap.
    legend_bottom = 0.26 if n_rows == 1 else 1.3 / figsize[1]
    fig.subplots_adjust(bottom=legend_bottom)
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        ncol=len(groups),
        frameon=False,
        title=group_column,
        title_fontsize=10,
        fontsize=10,
        handlelength=1.8,
        columnspacing=1.8,
        borderaxespad=0,
    )

    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()


In [ ]:
results_df = pd.read_csv("data/agg.csv")

In [ ]:
results_df_ai = results_df[results_df["Interviewer"] == "AI"]
results_df_human = results_df[results_df["Interviewer"] == "Human"]

# 1 - Users' Engagement

All analysis is restricted to the two interview sections that both conditions
administered. A third section was dropped from the human protocol after the
first day of data collection but remained in the AI-interviewer's
configuration, so it was put to 19 of 20 AI-led interviews and only 3 of 21
human-led ones. `agg.csv` is built on the shared sections alone, and the clock
is restricted to the same span, so duration and the rate measures cover
exactly the material being compared.

Because the two conditions spent comparable time on those sections but worked
through very different amounts of material, exposure is reported both as
totals and as rates per minute. The rates are the measure that does not depend
on interviews being of equal length — which they are not.

# Summary Statistics

In [ ]:
results_df.groupby("Interviewer")["question_count"].sum()

In [ ]:
# Group by 'Interviewer' and calculate summary statistics
summary_table = (
    results_df.groupby("Interviewer")[
        ["avg_response_length", "avg_response_duration_by_interviewer(sec)"]
    ]
    .agg(["mean", "median", "std", "min", "max"])
    .round(0)
)
summary_table.reset_index(inplace=True)
summary_table


In [ ]:
# Group by 'Interviewer' and calculate summary statistics
summary_table2 = (
    results_df.groupby("Interviewer")[["total_character_by_user", "question_count"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(1)
)
summary_table2.reset_index(inplace=True)
summary_table2


# Non-parametric Mann-Whitney U Test for Comparison

### Avg Response Duration

In [ ]:
res_dur_ai = results_df_ai["avg_response_duration_by_user(sec)"]
res_dur_human = results_df_human["avg_response_duration_by_user(sec)"]

mwu_res_dur = mwu(res_dur_ai, res_dur_human)
print(mwu_res_dur)

### Avg Response length

In [ ]:
res_length_ai = results_df_ai["avg_response_length"]
res_length_human = results_df_human["avg_response_length"]

mwu_res_length = mwu(res_length_ai, res_length_human)
print(mwu_res_length)

### Total Interview length by User 

In [ ]:
res_total_ai = results_df_ai["total_character_by_user"]
res_total_human = results_df_human["total_character_by_user"]

mwu_res_total = mwu(res_total_ai, res_total_human)
print(mwu_res_total)

### Question Count

In [ ]:
res_quest_ai = results_df_ai["question_count"]
res_quest_human = results_df_human["question_count"]

mwu_res_question = mwu(res_quest_ai, res_quest_human)
print(mwu_res_question)

### Question Formulation Time

In [ ]:
question_formulation_time_ai = results_df_ai["avg_response_duration_by_interviewer(sec)"]
question_formulation_time_human = results_df_human["avg_response_duration_by_interviewer(sec)"]

mwu_res_formulation = mwu(question_formulation_time_ai, question_formulation_time_human)
print(mwu_res_formulation)

In [ ]:
# Duration on the shared sections, and the two exposure rates. Reporting
# exposure per minute removes the objection that AI-led interviews simply ran
# longer, and does not rely on the interviews being of equal length.
mwu_duration = mwu(results_df_ai["duration(min)"], results_df_human["duration(min)"])
mwu_q_rate = mwu(results_df_ai["questions_per_min"], results_df_human["questions_per_min"])
mwu_c_rate = mwu(results_df_ai["characters_per_min"], results_df_human["characters_per_min"])

for name, res in [("Duration (min)", mwu_duration),
                  ("Questions per minute", mwu_q_rate),
                  ("Characters per minute", mwu_c_rate)]:
    print(f"{name}: U={res['U-val'].iloc[0]:.1f} p={res['p-val'].iloc[0]:.5f} "
          f"RBC={res['RBC'].iloc[0]:.3f}")

**Conclusion**: There is statistically significant difference between AI and human interviews in terms of average response duration, total interview length and question count. 

# 2- Relevance and Specificity

In [ ]:
grouped_df = pd.read_csv("data/relevance_and_specificity_scores.csv")

## Summary Statistics

In [ ]:
# Group by 'Interviewer' and calculate summary statistics
summary_table3 = (
    grouped_df.groupby("interviewer")[["relevance", "specificity"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(1)
)
summary_table3.reset_index(inplace=True)
summary_table3

In [ ]:
results_df = results_df.merge(
    grouped_df.astype({"conversation_id": int}).rename(
        columns={"interviewer": "Interviewer"}, inplace=False
    ),
    on=["conversation_id", "Interviewer"],
)

## Non-parametric Mann-Whitney U Test

In [ ]:
grouped_df_ai = grouped_df[grouped_df["interviewer"] == "AI"]
grouped_df_human = grouped_df[grouped_df["interviewer"] == "Human"]

In [ ]:
relevance_ai = grouped_df_ai["relevance"]
relevance_human = grouped_df_human["relevance"]

mwu_relevance = mwu(relevance_ai, relevance_human)
print(mwu_relevance)

In [ ]:
specificity_ai = grouped_df_ai["specificity"]
specificity_human = grouped_df_human["specificity"]

mwu_specificity = mwu(specificity_ai, specificity_human)
print(mwu_specificity)

In [ ]:
stat_table = pd.concat(
    [
        mwu_duration,
        mwu_res_length,
        mwu_res_total,
        mwu_res_question,
        mwu_q_rate,
        mwu_c_rate,
        mwu_res_formulation,
        # NOTE: this order must match `stat_table.index` below. Specificity and
        # relevance were previously concatenated the other way round, which
        # transposed the two rows of the reported table.
        mwu_specificity,
        mwu_relevance,
    ]
).drop("alternative", axis=1)

# Holm's step-down procedure holds the family-wise error rate at the same
# level as Bonferroni under no additional assumptions and is uniformly more
# powerful, so it is the primary correction. Several of these outcomes are
# algebraic functions of the same two quantities, which makes the single-step
# Bonferroni adjustment especially conservative; it is retained alongside so
# the two can be compared.
stat_table["p-val-raw"] = stat_table["p-val"].values
stat_table["p-val-bonferroni"] = multipletests(
    stat_table["p-val-raw"].values, method="bonferroni"
)[1]
stat_table["p-val"] = multipletests(stat_table["p-val-raw"].values, method="holm")[1]

stat_table.index = [
    "Duration",
    "Response Lengths",
    "Interview Length",
    "Questions per Interview",
    "Questions per Minute",
    "Characters per Minute",
    "Question Formulation Time",
    "Mean Specificity",
    "Mean Relevance",
]

# Guard against the labels drifting out of step with the concat order again.
for label, source in [("Duration", mwu_duration),
                      ("Questions per Minute", mwu_q_rate),
                      ("Characters per Minute", mwu_c_rate),
                      ("Mean Specificity", mwu_specificity),
                      ("Mean Relevance", mwu_relevance)]:
    assert stat_table.loc[label, "U-val"] == source["U-val"].iloc[0], label

print(
    stat_table.drop(columns="p-val-raw")
    .style.format(
        {
            "U-val": "{:.1f}",
            "p-val": "{:.3f}",
            "p-val-bonferroni": "{:.3f}",
            "RBC": "{:.3f}",
            "CLES": "{:.3f}",
        }
    )
    .to_latex()
)
stat_table.round(4)

## Plot

In [ ]:
# Revised plot configuration
plot_config = {
    "avg_response_length": {
        "title": "Response Lengths",
        "xlabel": "Mean Number of Characters per Question Answered\n(aggregated by interview)",
    },
    "total_character_by_user": {
        "title": "Interview Length",
        "xlabel": "Total Number of Characters in Answers",
    },
    "question_count": {
        "title": "Questions per Interview",
        "xlabel": "Number of Questions Answered",
    },
    "questions_per_min": {
        "title": "Questions per Minute",
        "xlabel": "Questions Answered per Minute",
    },
    "characters_per_min": {
        "title": "Characters per Minute",
        "xlabel": "Characters Produced per Minute",
    },
    "avg_response_duration_by_interviewer(sec)": {
        "title": "Question Formulation Time",
        "xlabel": "Seconds",
    },
    "specificity": {
        "title": "Mean Specificity",
        "xlabel": "Mean Specificity Score per Interview",
    },
    "relevance": {
        "title": "Mean Relevance",
        "xlabel": "Mean Relevance Score per Interview",
    },
}

# One figure for all eight panels: continuous panel labels and a single
# shared legend, rather than two images that each restart at A.
generate_violin_panels(results_df, stat_table, plot_config, "figures/plots.png")

# 3 - Effect sizes and equivalence bounds

A non-significant Mann-Whitney test does not establish that two groups are
equivalent. With 20 AI-led and 21 human-led interviews the study is only
powered to detect fairly large differences, so the informative quantity is not
the *p*-value but the range of differences the design can actually rule out.

For each outcome we report the Hodges-Lehmann shift estimate — the median of
all between-group pairwise differences, which is the location estimate the
Mann-Whitney test is built on — with a distribution-free confidence interval.
Differences outside the interval are inconsistent with the data; differences
inside it cannot be ruled out. The bound farthest from zero is therefore the
largest difference, in either direction, that the design cannot exclude.

Intervals are reported at 95% and, to stay consistent with the Bonferroni
correction applied to the tests, at a family-wise 1 - 0.05/9 level.

In [ ]:
from scipy import stats


def hodges_lehmann(ai, human, conf=0.95):
    """Hodges-Lehmann shift (AI - human) with a distribution-free CI.

    The interval is the Moses interval: the pairwise differences are sorted and
    the bounds are taken at the order statistics given by the normal
    approximation to the Mann-Whitney null distribution.
    """
    ai = np.asarray(ai, dtype=float)
    human = np.asarray(human, dtype=float)
    diffs = np.sort((ai[:, None] - human[None, :]).ravel())

    m, n = len(ai), len(human)
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    k = m * n / 2 - z * np.sqrt(m * n * (m + n + 1) / 12)
    k = int(np.floor(k))

    if k < 0:  # interval wider than the data support
        return float(np.median(diffs)), -np.inf, np.inf
    lo = diffs[k]
    hi = diffs[len(diffs) - 1 - k]
    return float(np.median(diffs)), float(lo), float(hi)


OUTCOMES = {
    "Duration": "duration(min)",
    "Response Lengths": "avg_response_length",
    "Interview Length": "total_character_by_user",
    "Questions per Interview": "question_count",
    "Questions per Minute": "questions_per_min",
    "Characters per Minute": "characters_per_min",
    "Question Formulation Time": "avg_response_duration_by_interviewer(sec)",
    "Mean Specificity": "specificity",
    "Mean Relevance": "relevance",
}

FAMILY_CONF = 1 - 0.05 / len(OUTCOMES)

rows = []
for label, column in OUTCOMES.items():
    ai = results_df.loc[results_df["Interviewer"] == "AI", column].dropna()
    human = results_df.loc[results_df["Interviewer"] == "Human", column].dropna()

    shift, lo, hi = hodges_lehmann(ai, human)
    _, lo_fw, hi_fw = hodges_lehmann(ai, human, conf=FAMILY_CONF)

    rows.append(
        {
            "Outcome": label,
            "AI mean": ai.mean(),
            "Human mean": human.mean(),
            "HL shift (AI - human)": shift,
            "95% CI low": lo,
            "95% CI high": hi,
            f"{FAMILY_CONF:.1%} CI low": lo_fw,
            f"{FAMILY_CONF:.1%} CI high": hi_fw,
            # Largest difference compatible with the data, either direction.
            "Excludes |diff| >": max(abs(lo_fw), abs(hi_fw)),
        }
    )

equivalence_table = pd.DataFrame(rows).set_index("Outcome")
equivalence_table.round(3)

# 4 - Inter-rater reliability

Every coded question-answer pair was scored independently by two coders; the
score used in the analysis above is the mean of the two. `coder_agreement.csv`
holds both coders' scores per pair (no response text, to preserve
confidentiality).

Because specificity and relevance are ordinal, we report Krippendorff's alpha
with the ordinal difference function alongside quadratically weighted Cohen's
kappa, exact agreement, and agreement within one scale point.

In [ ]:
from sklearn.metrics import cohen_kappa_score

coder_df = pd.read_csv("data/coder_agreement.csv")


def krippendorff_alpha_ordinal(units):
    """Krippendorff's alpha with the ordinal difference function.

    `units` is an (n_units, n_coders) array of integer ranks; no missing data.
    Follows Krippendorff (2012), ch. 12: alpha = 1 - D_observed / D_expected,
    where the ordinal metric weights a disagreement by the number of ranks
    spanned rather than treating all disagreements alike.
    """
    units = np.asarray(units, dtype=int)
    values = np.unique(units)
    index = {v: i for i, v in enumerate(values)}
    n_values = len(values)

    # Marginal frequency of each value across all coder-unit entries.
    marginal = np.zeros(n_values)
    for v in units.ravel():
        marginal[index[v]] += 1
    n_total = marginal.sum()

    # Ordinal difference function: the squared span between two ranks,
    # measured in marginal mass, with the endpoints counted half.
    delta = np.zeros((n_values, n_values))
    for c in range(n_values):
        for k in range(c, n_values):
            span = marginal[c:k + 1].sum() - (marginal[c] + marginal[k]) / 2
            delta[c, k] = delta[k, c] = span ** 2

    # Observed disagreement, averaged over units.
    d_observed = sum(
        sum(delta[index[a], index[b]] for a in row for b in row) / (len(row) - 1)
        for row in units
    ) / n_total

    # Expected disagreement from the marginal distribution alone.
    d_expected = sum(
        marginal[c] * marginal[k] * delta[c, k]
        for c in range(n_values)
        for k in range(n_values)
    ) / (n_total * (n_total - 1))

    return 1 - d_observed / d_expected


reliability_rows = []
for measure in ["specificity", "relevance"]:
    a = coder_df[f"{measure}_coder_a"]
    b = coder_df[f"{measure}_coder_b"]
    both = a.notna() & b.notna()
    a, b = a[both].astype(int), b[both].astype(int)

    reliability_rows.append(
        {
            "Measure": measure.capitalize(),
            "n pairs": int(both.sum()),
            "Exact agreement": (a == b).mean(),
            "Within 1 point": ((a - b).abs() <= 1).mean(),
            "Krippendorff alpha (ordinal)": krippendorff_alpha_ordinal(
                np.column_stack([a, b])
            ),
            "Cohen kappa (quadratic)": cohen_kappa_score(a, b, weights="quadratic"),
            "Cohen kappa (linear)": cohen_kappa_score(a, b, weights="linear"),
        }
    )

reliability_table = pd.DataFrame(reliability_rows).set_index("Measure")
reliability_table.round(3)

# 5 - Regression estimates

The non-parametric tests are supplemented with OLS estimates of the condition
difference. Two specifications are reported per outcome:

- **Model 1** regresses the outcome on the condition indicator alone. Because
  assignment was randomised, this is the estimate the design licenses.
- **Model 2** adds fixed effects for the day of interview, since randomisation
  was carried out separately within each day. `interview_day.csv` maps each
  interview to a day index; the index is derived from message timestamps and
  deliberately carries no calendar date. Two of the ten days contain a single
  interview, which a day fixed effect fits exactly, so Model 2 is reported with
  classical standard errors only.

Respondent covariates (age, gender, country of origin, study programme, degree
level, prior ChatGPT use) are deliberately **not** included. Two reasons. They
are respondent-level personal data that cannot be released without serious
re-identification risk: in a sample this small, 85% of participants are unique
on that combination of characteristics, and the repository already publishes an
interview identifier they could be joined on. And with 41 interviews, a
specification carrying that many dummies leaves too few residual degrees of
freedom to support the coefficients it produces. Sample composition is instead
reported as marginal counts in the balance table in the paper.

Heteroskedasticity-consistent (HC3) standard errors are reported alongside the
classical ones wherever they are defined, since the two groups differ markedly
in variance.

In [ ]:
import statsmodels.api as sm

day_df = pd.read_csv("data/interview_day.csv")
reg_df = results_df.merge(day_df, on="conversation_id", how="left")
assert reg_df["interview_day"].notna().all(), "every interview needs a day index"

# AI is the treatment; the human-interviewer condition is the reference.
reg_df["ai"] = (reg_df["Interviewer"] == "AI").astype(float)

# Two of the ten days contain a single interview. A day fixed effect fits those
# observations exactly, giving them leverage 1, which leaves HC3 undefined
# (its residual correction divides by (1 - h)^2). We report the classical
# standard error in that case rather than a spurious one.
singleton_days = sorted(
    reg_df.interview_day.value_counts()[lambda s: s == 1].index.tolist()
)
print(f"days containing a single interview: {singleton_days}")

REG_OUTCOMES = {
    "Interview length": "total_character_by_user",
    "Response length": "avg_response_length",
    "Question count": "question_count",
    "Specificity": "specificity",
    "Relevance": "relevance",
}


def fit_condition_model(outcome, with_day_fe):
    y = reg_df[outcome]
    X = reg_df[["ai"]]
    if with_day_fe:
        X = X.join(pd.get_dummies(reg_df["interview_day"], prefix="day", drop_first=True))
    X = sm.add_constant(X).astype(float)

    model = sm.OLS(y, X).fit()

    # HC3 is only meaningful while every observation has leverage below 1.
    leverage = model.get_influence().hat_matrix_diag
    if leverage.max() < 1 - 1e-8:
        robust = model.get_robustcov_results(cov_type="HC3")
        position = list(X.columns).index("ai")
        se_hc3, p_hc3 = robust.bse[position], robust.pvalues[position]
    else:
        se_hc3 = p_hc3 = np.nan

    return {
        "n": int(model.nobs),
        "params": int(model.df_model) + 1,
        "df resid": int(model.df_resid),
        "coef": model.params["ai"],
        "se": model.bse["ai"],
        "p": model.pvalues["ai"],
        "se (HC3)": se_hc3,
        "p (HC3)": p_hc3,
        "R2": model.rsquared,
        "adj R2": model.rsquared_adj,
        "model F p": model.f_pvalue,
    }


regression_table = pd.DataFrame(
    [
        {"Outcome": label, "Model": "1: condition only" if not fe else "2: + day FE",
         **fit_condition_model(column, fe)}
        for label, column in REG_OUTCOMES.items()
        for fe in (False, True)
    ]
).set_index(["Outcome", "Model"])

regression_table.round(4)